# Bayesian Belief Updating for Diagnostic Tests (with Prior Uncertainty)

**Goal.** Compute the probability of having a disease after a **positive** or **negative** test while modeling our **uncertainty about prevalence** (and optionally test sensitivity/specificity) using Bayesian priors. We work from first principles using NumPy and Matplotlib.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports & Helper Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

def ci_quantiles(x, qs=(0.025, 0.5, 0.975)):
    x = np.asarray(x)
    return np.quantile(x, qs)

def show_hist_and_cdf(samples, title="Posterior Probability", bins=80):
    samples = np.clip(samples, 0, 1)
    fig = plt.figure(figsize=(10,4))
    # Histogram
    ax1 = plt.subplot(1,2,1)
    ax1.hist(samples, bins=bins, density=True)
    ax1.set_title(title + " — PDF (MC)")
    ax1.set_xlabel("Probability"); ax1.set_ylabel("Density")
    # CDF
    ax2 = plt.subplot(1,2,2)
    xs = np.sort(samples)
    ys = np.linspace(0,1,len(xs))
    ax2.plot(xs, ys)
    ax2.set_title(title + " — CDF (MC)")
    ax2.set_xlabel("Probability"); ax2.set_ylabel("Cumulative")
    plt.tight_layout(); plt.show()

## 2) Bayesian Setup and Closed-Form (Point Prevalence)

Let $D$ be disease (1=yes, 0=no), and $T$ be a test result (+ or −). For **point** prevalence $\pi$, sensitivity $s=\Pr(T{=}+\mid D)$, and specificity $c=\Pr(T{-}\mid \neg D)$:

- **Positive test (PPV)**: $\Pr(D\mid T{=}+) = \dfrac{s\,\pi}{s\,\pi + (1-c)\,(1-\pi)}$.
- **Negative test (1−NPV)**: $\Pr(D\mid T{-}) = \dfrac{(1-s)\,\pi}{(1-s)\,\pi + c\,(1-\pi)}$.

**Uncertain prevalence.** If we are uncertain about $\pi$ and model it with a prior distribution (e.g., $\pi\sim\text{Beta}(a,b)$), then the **posterior disease probability for a single patient** becomes a *random variable* obtained by plugging each plausible $\pi$ into the above formulas. We approximate its distribution via Monte Carlo sampling. We can also model uncertainty in sensitivity/specificity with Beta priors if desired.

## 3) Inputs — Priors and Monte Carlo Settings

Set Beta parameters for prevalence and (optionally) for sensitivity and specificity. If you prefer fixed $s,c$, set `S_BETA=None` or `C_BETA=None`.

In [ ]:
# --- Prior over prevalence π ~ Beta(A,B) ---
# Example: a rare disease with mean prevalence ≈ 1% and moderate uncertainty.
A, B = 1.0, 99.0   # mean = A/(A+B) = 0.01

# --- Sensitivity & Specificity ---
# You can specify fixed values or Beta priors for uncertainty.
# Fixed example:
S_FIXED = 0.95     # sensitivity
C_FIXED = 0.98     # specificity

# Or, use Beta priors to reflect uncertainty (set to None to use fixed):
S_BETA = None      # e.g., (95, 5) centers ~0.95 with moderate confidence
C_BETA = None      # e.g., (98, 2) centers ~0.98 with high confidence

# Monte Carlo samples
N_SAMPLES = 100_000
print("Configured. E[π] =", A/(A+B))

## 4) Monte Carlo Sampling of Posterior Disease Probabilities

In [ ]:
def sample_prevalence(a, b, n, rng):
    return rng.beta(a, b, size=n)

def sample_param_fixed_or_beta(fixed, beta_params, n, rng):
    if beta_params is None:
        return np.full(n, fixed, dtype=float)
    else:
        a, b = beta_params
        return rng.beta(a, b, size=n)

def posterior_prob_positive(pi, sens, spec):
    # PPV: P(D=1 | T=+) = (sens*pi) / (sens*pi + (1-spec)*(1-pi))
    num = sens * pi
    den = sens * pi + (1.0 - spec) * (1.0 - pi)
    return num / np.maximum(den, 1e-15)

def posterior_prob_negative(pi, sens, spec):
    # P(D=1 | T=-) = ((1-sens)*pi) / ((1-sens)*pi + spec*(1-pi))
    num = (1.0 - sens) * pi
    den = (1.0 - sens) * pi + spec * (1.0 - pi)
    return num / np.maximum(den, 1e-15)

# Draw samples
pi_samps   = sample_prevalence(A, B, N_SAMPLES, rng)
s_samps    = sample_param_fixed_or_beta(S_FIXED, S_BETA, N_SAMPLES, rng)
c_samps    = sample_param_fixed_or_beta(C_FIXED, C_BETA, N_SAMPLES, rng)

ppv_samps  = posterior_prob_positive(pi_samps, s_samps, c_samps)
neg_samps  = posterior_prob_negative(pi_samps, s_samps, c_samps)

# Point-prevalence comparison (using mean prevalence and mean s,c)
pi_mean = A/(A+B)
s_mean  = (S_FIXED if S_BETA is None else (S_BETA[0]/(S_BETA[0]+S_BETA[1])))
c_mean  = (C_FIXED if C_BETA is None else (C_BETA[0]/(C_BETA[0]+C_BETA[1])))
ppv_point = posterior_prob_positive(pi_mean, s_mean, c_mean)
neg_point = posterior_prob_negative(pi_mean, s_mean, c_mean)

print(f"Point estimate PPV at mean params: {ppv_point:.4f}")
print(f"Point estimate P(D=1 | T=-) at mean params: {neg_point:.4f}")

## 5) Visualize Posterior Disease Probability Distributions

In [ ]:
show_hist_and_cdf(ppv_samps, title="P(D=1 | T=+)")
show_hist_and_cdf(neg_samps, title="P(D=1 | T=-)")

## 6) Credible Intervals and Summary Statistics

In [ ]:
def summarize(name, samples, point=None):
    lo, med, hi = ci_quantiles(samples, (0.025, 0.5, 0.975))
    mean = float(np.mean(samples))
    print(f"{name}:")
    print(f"  mean={mean:.4f}  median={med:.4f}  95% CI=({lo:.4f}, {hi:.4f})")
    if point is not None:
        print(f"  point-estimate (at mean params) = {point:.4f}")
    print()

summarize("P(D=1 | T=+)", ppv_samps, ppv_point)
summarize("P(D=1 | T=-)", neg_samps, neg_point)

## 7) Sensitivity Analysis: Vary Mean Prevalence

We sweep the **mean prevalence** while holding sensitivity/specificity fixed at their means to show how PPV and P(D|−) change.

In [ ]:
means = np.linspace(1e-4, 0.2, 60)  # from 0.01% to 20%
ppv_curve = posterior_prob_positive(means, s_mean, c_mean)
neg_curve = posterior_prob_negative(means, s_mean, c_mean)

fig = plt.figure(figsize=(6,4))
plt.plot(means, ppv_curve, label="P(D=1 | T=+)")
plt.plot(means, neg_curve, label="P(D=1 | T=-)")
plt.xlabel("Mean prevalence π"); plt.ylabel("Posterior probability")
plt.title("Effect of Prevalence on Posterior Probabilities")
plt.legend(); plt.tight_layout(); plt.show()

## 8) Notes on Modeling Choices

- For a **single patient’s** test result, integrating over a Beta prior for prevalence effectively plugs in the **mean prevalence** if sensitivity/specificity are fixed, because Bayes’ rule is linear in $\pi$. Using a prior distribution is still useful when we propagate **uncertainty** (via Monte Carlo) or when we also put priors over **sensitivity/specificity**.
- To learn prevalence from **multiple** tests across a cohort, extend the model to update the Beta prior with observed positives/negatives (accounting for test errors). Then recompute individual posterior probabilities with the updated prevalence distribution.
- If the test is used repeatedly or combined with other evidence (symptoms, risk factors), the posterior after one piece of evidence becomes the prior for the next (sequential Bayes).

## 9) Save Artifacts & Download

We save Monte Carlo samples, summaries, and plots. Use the helper below to download a ZIP in Colab.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/posteriors_samples.npz",
         ppv=ppv_samps, neg=neg_samps,
         pi_samps=pi_samps, s_samps=s_samps, c_samps=c_samps,
         meta=np.array([A, B, s_mean, c_mean, N_SAMPLES], dtype=float))

# Minimal text summary
summary = {}
for name, samples, point in [
    ("P(D=1 | T=+)", ppv_samps, ppv_point),
    ("P(D=1 | T=-)", neg_samps, neg_point)
]:
    lo, med, hi = ci_quantiles(samples, (0.025, 0.5, 0.975))
    summary[name] = {"mean": float(np.mean(samples)),
                     "median": float(med),
                     "ci95": [float(lo), float(hi)],
                     "point_estimate": float(point)}
with open("artifacts/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 10) Exercises

- Start with a different prior (e.g., informative Beta reflecting a subpopulation). Compare results with and without uncertainty in sensitivity/specificity.
- Add a second independent test and do **sequential updating**: posterior from Test 1 becomes prior input to Test 2.
- Turn this into a small function `bayes_ppv(prior, sens, spec)` that accepts either scalars or Beta hyperparameters and returns both point and distributional summaries.